# FHIR-Daten mit Pathling analysieren

## 1. Installation

### 1.1 Python-Pakete


In [ ]:
%pip install pyspark==4.0.4 pandas
%pip install --no-deps pathling==10.0.0

### 1.2 Java 21

In [ ]:
%%bash
apt-get update
DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends openjdk-21-jre-headless tree

### 1.3 Synthea und DuckDB-CLI

In [ ]:
%%bash
set -euo pipefail
mkdir -p "$PWD/tools" && cd "$PWD/tools"

SYNTHEA_VERSION=v4.0.0
SYNTHEA_SHA256=ed43c20ad40ba5c3bc724503a5af032715fe3c491620b766148e7c2361e6ecc1

curl -fsSL -o synthea-with-dependencies.jar \
  "https://github.com/synthetichealth/synthea/releases/download/${SYNTHEA_VERSION}/synthea-with-dependencies.jar"
echo "${SYNTHEA_SHA256}  synthea-with-dependencies.jar" | sha256sum -c - \
  || { rm -f synthea-with-dependencies.jar; exit 1; }

In [ ]:
%%bash
set -euo pipefail
mkdir -p "$PWD/tools" && cd "$PWD/tools"

DUCKDB_VERSION=v2.0.0
DUCKDB_SHA256=c6507206baf7793c6013dbd82368469a45f20dc0e17a6f92f204602f652ed2bc

curl -fsSL -o duckdb-cli-linux-amd64.tar.gz \
  "https://github.com/duckdb/duckdb/releases/download/${DUCKDB_VERSION}/duckdb-cli-linux-amd64.tar.gz"
echo "${DUCKDB_SHA256}  duckdb-cli-linux-amd64.tar.gz" | sha256sum -c - \
  || { rm -f duckdb-cli-linux-amd64.tar.gz; exit 1; }

tar -xzf duckdb-cli-linux-amd64.tar.gz duckdb
./duckdb --version
rm duckdb-cli-linux-amd64.tar.gz

## 2. Synthetische Patienten mit Synthea erzeugen

In [ ]:
%%bash
set -euo pipefail
rm -rf synthea

java -Duser.timezone=UTC -jar tools/synthea-with-dependencies.jar \
    -p 150 -o false -s 20261029 -cs 20261029 -r 20261029 -e 20261029 \
    --exporter.baseDirectory=synthea \
    --exporter.fhir.export=true \
    --exporter.fhir.included_resources=Patient,Condition,Observation,Encounter,Procedure,MedicationRequest

## 3. Bundles mit Pathling einlesen

`PathlingContext.create()` startet eine lokale Spark-Session. Beim ersten Mal lädt Spark die JARs von Pathling und Delta Lake aus Maven Central herunter.

- `enable_extensions=True` behält FHIR-Extensions, sodass sie abgefragt werden können.
- `enable_terminology=False`: Dieses Notebook nutzt die Terminologie-Funktionen von Pathling nicht und braucht daher keinen Terminologieserver.

⚠️ Siehst du den Fehler `Exception in thread "main" java.lang.UnsupportedOperationException: getSubject is not supported` hast du die falsche Java Version installiert. Pathling benötigt Java 21.

In [ ]:
import json
from pathlib import Path

from pathling import PathlingContext
from pyspark.sql import SparkSession

BUNDLES_DIR = Path.cwd() / "synthea" / "fhir"

spark = SparkSession.getActiveSession()
pc = PathlingContext.create(
    spark,
    spark_conf=None if spark else {"spark.driver.memory": "6g"},
    enable_extensions=True,
    enable_terminology=False,
)
pc.spark.sparkContext.setLogLevel("ERROR")
print("Spark", pc.spark.version)

`pc.read.bundles()` extrahiert die angegebenen Ressourcentypen aus allen Bundles im Verzeichnis. Jeder Typ wird zu einem Spark-DataFrame (so etwas wie eine Tabelle), dessen verschachtelte Spalten die Struktur der FHIR-Ressource abbilden.

In [ ]:
RESOURCE_TYPES = [
    "Patient",
    "Condition",
    "Encounter",
    "Observation",
    "Procedure",
    "MedicationRequest",
]

data = pc.read.bundles(str(BUNDLES_DIR), resource_types=RESOURCE_TYPES)

# Zeige die Anzahl der Ressourcen pro Type an
pc.spark.createDataFrame(
    [
        (resource_type, data.read(resource_type).count())
        for resource_type in data.resource_types()
    ],
    ["resource_type", "resources"],
).orderBy("resources", ascending=False).show()

Das eingelesene Patient-DataFrame hat eine Spalte pro FHIR-Element. Wiederholbare Elemente wie `name` sind Arrays von Structs:

In [ ]:
patient_df = data.read("Patient")
print(f"{len(patient_df.columns)} Spalten: {', '.join(patient_df.columns)}\n")
patient_df.select("id", "gender", "birthDate", "name").show(3, truncate=110)

In [ ]:
data.read("Condition").select("id", "subject.reference", "code").show(3, truncate=False)

## 4. ViewDefinition 1: Patient → Tabelle

Eine [ViewDefinition](https://build.fhir.org/ig/HL7/sql-on-fhir/StructureDefinition-ViewDefinition.html) beschreibt, wie ein Ressourcentyp in eine Tabelle mit ausgewählten Spalten überführt wird:

- `resource`: der Ressourcentyp. Die View liefert eine Zeile pro Ressource, außer ein `forEach` entschachtelt ein wiederholbares Element.
- `select[].column[]`: Jede Spalte hat einen Namen (`name`) und einen [FHIRPath](https://pathling.csiro.au/docs/libraries/fhirpath)-Ausdruck (`path`), der auf der Ressource ausgewertet wird.

Die View unten nutzt einige FHIRPath-Funktionen:

| Ausdruck | Zweck |
|---|---|
| `getResourceKey()` | Schlüssel der Ressource für Joins (siehe Abschnitt 6) |
| `name.where(use = 'official')` | Ein Element eines Arrays filtern |
| `given.join(' ')` | Eine Liste zu einer Zeichenkette zusammenfügen |
| `deceased.ofType(dateTime)` | Einen Typ eines Choice-Elements (`deceased[x]`) auswählen |
| `extension('…')` | In Extensions navigieren |

Eine ViewDefinition ist reines JSON, dieselbe Definition läuft daher auf jeder SQL-on-FHIR-Implementierung. Hier ist sie ein Python-Dictionary, das wir serialisieren und an `data.view()` übergeben.

In [ ]:
patient_view = {
    "resourceType": "ViewDefinition",
    "name": "patient_demographics",
    "status": "active",
    "resource": "Patient",
    "select": [
        {
            "column": [
                {"name": "patient_key", "path": "getResourceKey()"},
                {"name": "gender", "path": "gender"},
                {"name": "birth_date", "path": "birthDate"},
                {
                    "name": "is_deceased",
                    "path": "deceased.ofType(dateTime).exists() or deceased.ofType(boolean).anyTrue()",
                },
                {"name": "deceased_at", "path": "deceased.ofType(dateTime)"},
                {
                    "name": "family_name",
                    "path": "name.where(use = 'official').family.first()",
                },
                {
                    "name": "given_names",
                    "path": "name.where(use = 'official').given.join(' ')",
                },
                {"name": "city", "path": "address.first().city"},
            ]
        }
    ],
}

# view() ist eine Pathling-spezifische Operation
patients = data.view(json=json.dumps(patient_view))
patients.printSchema()
patients.limit(10).toPandas()

## 5. ViewDefinition 2: Condition → Tabelle

Diese View hat ein zweites `select` mit `forEach: "code.coding"`, das für jedes Coding im Code der Condition eine Zeile erzeugt. Die Spalten aus dem ersten `select` werden in jeder dieser Zeilen wiederholt.

`subject.getReferenceKey(Patient)` liefert einen Schlüssel, der zum `getResourceKey()` der referenzierten Patient-Ressource passt. Mit diesen beiden Funktionen joined man in SQL on FHIR Views.

In [ ]:
condition_view = {
    "resourceType": "ViewDefinition",
    "name": "condition_codes",
    "status": "active",
    "resource": "Condition",
    "select": [
        {
            "column": [
                {"name": "condition_key", "path": "getResourceKey()"},
                {"name": "patient_key", "path": "subject.getReferenceKey(Patient)"},
                {
                    "name": "encounter_key",
                    "path": "encounter.getReferenceKey(Encounter)",
                },
                {
                    "name": "clinical_status",
                    "path": "clinicalStatus.coding.code.first()",
                },
                {"name": "onset", "path": "onset.ofType(dateTime)"},
                {"name": "abatement", "path": "abatement.ofType(dateTime)"},
            ]
        },
        {
            "forEach": "code.coding",
            "column": [
                {"name": "code_system", "path": "system"},
                {"name": "code", "path": "code"},
                {"name": "display", "path": "display"},
            ],
        },
    ],
}

conditions = data.view(json=json.dumps(condition_view))
print(f"{conditions.count()} Zeilen")
conditions.limit(10).toPandas()

Synthea kodiert die meisten Erkrankungen in SNOMED CT und einige (vor allem Krebserkrankungen) in ICD-10:

In [ ]:
conditions.groupBy("code_system").count().toPandas()

## 6. Patienten und Diagnosen verknüpfen

Die Ergebnisse der Views sind gewöhnliche Spark-DataFrames. Wir verknüpfen sie über `patient_key`, zuerst mit der DataFrame-API und darunter mit Spark SQL über temporäre Views.

In [ ]:
from pyspark.sql import functions as F

(
    patients.join(conditions, "patient_key")
    .withColumn("onset_date", F.substring("onset", 1, 10))
    .withColumn(
        "age_at_onset",
        F.floor(
            F.months_between(F.to_date("onset_date"), F.to_date("birth_date")) / 12
        ),
    )
    .orderBy("family_name", "given_names", "onset")
    .select(
        "given_names",
        "family_name",
        "gender",
        "birth_date",
        F.col("display").alias("condition"),
        "clinical_status",
        "onset_date",
        "age_at_onset",
    )
    .limit(15)
    .toPandas()
)

In [ ]:
# erzeugt in Spark SQL eine temporäre View,
# die wir in SQL-Abfragen referenzieren können:
patients.createOrReplaceTempView("patients")
conditions.createOrReplaceTempView("conditions")

pc.spark.sql('''
    SELECT p.given_names,
           p.family_name,
           p.gender,
           p.birth_date,
           c.display AS condition,
           c.clinical_status,
           left(c.onset, 10) AS onset_date,
           floor(months_between(to_date(left(c.onset, 10)), to_date(p.birth_date)) / 12) AS age_at_onset
    FROM patients p -- hier wird die oben registrierte View referenziert
    JOIN conditions c ON c.patient_key = p.patient_key
    ORDER BY p.family_name, p.given_names, c.onset
''').limit(15).toPandas()

Die zehn häufigsten Erkrankungen der Kohorte, nach Anzahl der betroffenen Personen:

In [ ]:
TOP_CONDITIONS_SQL = '''
    SELECT c.display AS condition,
           count(DISTINCT p.patient_key) AS patients,
           round(100 * count(DISTINCT p.patient_key) / (SELECT count(*) FROM patients), 1) AS pct_of_cohort,
           count(DISTINCT IF(p.gender = 'female', p.patient_key, NULL)) AS female,
           count(DISTINCT IF(p.gender = 'male', p.patient_key, NULL)) AS male,
           round(avg(months_between(to_date(left(c.onset, 10)), to_date(p.birth_date)) / 12), 1) AS mean_age_at_onset
    FROM patients p
    JOIN conditions c ON c.patient_key = p.patient_key
    WHERE c.code_system = 'http://snomed.info/sct'
    GROUP BY c.display
    ORDER BY patients DESC, condition
    LIMIT 10
'''

pc.spark.sql(TOP_CONDITIONS_SQL).toPandas()

## 7. FHIR-Ressourcen als Delta-Lake-Tabellen speichern

`data.write.delta()` schreibt für jeden Ressourcentyp eine [Delta-Lake](https://delta.io)-Tabelle namens `<Ressourcentyp>.parquet` in ein Verzeichnis. Jede Tabelle besteht aus Parquet-Dateien und dem Transaktionslog `_delta_log/`.

In [ ]:
DELTA_DIR = Path.cwd() / "delta"

result = data.write.delta(str(DELTA_DIR))

Auflisten der Tabellen als Baum-Struktur:

In [ ]:
%%bash
tree ./delta/

Pathling kann die Tabellen wieder als Datenquelle einlesen, und dieselbe ViewDefinition läuft unverändert darauf:

In [ ]:
delta_data = pc.read.delta(str(DELTA_DIR))
delta_data.view(json=json.dumps(patient_view)).limit(5).toPandas()

### JSON vs Delta Lake

Wir führen beide ViewDefinitions und die Top-10-Abfrage aus Abschnitt 6 einmal auf den JSON-Bundles und einmal auf den Delta-Tabellen aus. Bei den Bundles muss Spark für jede Abfrage alle JSON-Dateien erneut lesen und parsen. Bei Delta liest es die relevanten Spalten der Parquet-Dateien.

In [ ]:
import time


def views_and_top_conditions(source):
    source.view(json=json.dumps(patient_view)).createOrReplaceTempView("patients")
    source.view(json=json.dumps(condition_view)).createOrReplaceTempView("conditions")
    return pc.spark.sql(TOP_CONDITIONS_SQL).toPandas()


results, seconds = {}, {}
for label, source in [("JSON-Bundles", data), ("Delta Lake", delta_data)]:
    start = time.perf_counter()
    results[label] = views_and_top_conditions(source)
    seconds[label] = time.perf_counter() - start
    print(f"{label:12} {seconds[label]:6.1f} s")

print(
    f"\nDelta Lake war {seconds['JSON-Bundles'] / seconds['Delta Lake']:.0f}-mal so schnell."
)
print("Identische Ergebnisse:", results["JSON-Bundles"].equals(results["Delta Lake"]))

## 8. Die Delta-Tabellen mit der DuckDB-CLI abfragen

Delta Lake ist ein offenes Format, andere Engines können lesen, was Pathling geschrieben hat. DuckDB liest Delta-Tabellen über seine [`delta`-Erweiterung](https://duckdb.org/docs/stable/core_extensions/delta).

- Jede Tabelle wird über ihren Pfad gelesen: `delta_scan('delta/<Ressourcentyp>.parquet')`.
- `.maxwidth 140` begrenzt die Breite der Ergebnistabellen auf die Breite der Notebook-Ausgabe.

In [ ]:
%%bash
tools/duckdb <<'SQL'
-- Die delta-Erweiterung installieren. Danach lädt DuckDB sie automatisch, sobald delta_scan() vorkommt.
INSTALL delta;

SELECT 'Patient' AS resource_type, count(*) AS resources FROM delta_scan('delta/Patient.parquet')
UNION ALL
SELECT 'Condition', count(*) FROM delta_scan('delta/Condition.parquet')
UNION ALL
SELECT 'Observation', count(*) FROM delta_scan('delta/Observation.parquet');
SQL

### Pathling FHIR Schema

Jedes FHIR-Element ist eine Spalte. Komplexe Datentypen wie `HumanName` werden zu `STRUCT`s, wiederholbare Elemente zu Arrays (`[]`). Pathling ergänzt einige technische Spalten, die keine FHIR-Elemente sind:

- `id_versioned`: die Ressourcen-ID samt Typ (und ggf. Version).
- `_fid`: eine eindeutige ID für jedes Element, über die Extensions zugeordnet werden.
- `_extension`: eine Map von der `_fid` eines Elements auf dessen Extensions.

Choice-Elemente wie `deceased[x]` werden zu einer Spalte pro Datentyp (`deceasedBoolean`, `deceasedDateTime`).

In [ ]:
%%bash
tools/duckdb <<'SQL'
.maxwidth 140
SELECT column_name, column_type
FROM (DESCRIBE SELECT * FROM delta_scan('delta/Patient.parquet'));
SQL

In [ ]:
%%bash
tools/duckdb <<'SQL'
.maxwidth 140
-- name ist eine Liste von HumanName-Structs: unnest() erzeugt eine Zeile pro Name.
SELECT p.id, n.use, n.family, n.given, n.prefix
FROM delta_scan('delta/Patient.parquet') AS p, unnest(p.name) AS t(n)
LIMIT 6;
SQL

In [ ]:
%%bash
tools/duckdb <<'SQL'
.maxwidth 140
-- Struct-Felder spricht man mit Punkten an. Listen sind in DuckDB 1-basiert.
SELECT subject.reference      AS subject,
       code.coding[1].code    AS code,
       code.coding[1].display AS display,
       onsetDateTime
FROM delta_scan('delta/Condition.parquet')
LIMIT 5;
SQL

### Der Join aus Abschnitt 6 auf den rohen Ressourcen

Dieselbe Top-10-Frage, direkt auf den verschachtelten Tabellen formuliert. Ohne ViewDefinition muss die Abfrage die Codings selbst entschachteln und wissen, dass Referenzen als `Patient/<id>` gespeichert sind. Die Zahlen stimmen mit dem Spark-Ergebnis aus Abschnitt 6 überein.

In [ ]:
%%bash
tools/duckdb <<'SQL'
.maxwidth 140
SELECT coding.display                                    AS condition,
       count(DISTINCT p.id)                              AS patients,
       count(DISTINCT p.id) FILTER (p.gender = 'female') AS female,
       count(DISTINCT p.id) FILTER (p.gender = 'male')   AS male
FROM delta_scan('delta/Condition.parquet') AS c
CROSS JOIN unnest(c.code.coding) AS t(coding)
JOIN delta_scan('delta/Patient.parquet') AS p ON c.subject.reference = 'Patient/' || p.id
WHERE coding.system = 'http://snomed.info/sct' AND coding.display LIKE '%(disorder)'
GROUP BY ALL
ORDER BY patients DESC, condition
LIMIT 10;
SQL

### Blick in eine Delta-Tabelle

Eine Delta-Tabelle ist ein Verzeichnis mit Parquet-Dateien und einem Transaktionslog aus JSON-Commits in `_delta_log/`. DuckDB liest das Log wie jede andere JSON-Datei.

In [ ]:
%%bash
tools/duckdb <<'SQL'
.maxwidth 140
SELECT file FROM glob('delta/Patient.parquet/**') WHERE file NOT LIKE '%.crc';
SQL

In [ ]:
%%bash
cd "$WORK_DIR"
tools/duckdb <<'SQL'
.maxwidth 140
-- Jede Zeile einer Commit-Datei ist eine Aktion. Der erste Commit enthält die Operation,
-- die Protokollversionen, die Leser und Schreiber unterstützen müssen, die Metadaten der Tabelle ...
SELECT commitInfo.operation,
       protocol.minReaderVersion,
       protocol.minWriterVersion,
       metaData.id AS table_id
FROM read_json_auto('delta/Patient.parquet/_delta_log/00000000000000000000.json')
WHERE add IS NULL;

-- ... und eine "add"-Aktion pro Parquet-Datei, mit Statistiken.
SELECT add.path,
       add.size AS bytes,
       json_extract(add.stats, '$.numRecords') AS records
FROM read_json_auto('delta/Patient.parquet/_delta_log/00000000000000000000.json')
WHERE add IS NOT NULL;
SQL

## Ideen zum Ausprobieren:

1. Ergänze die Condition-View um eine `where`-Klausel, sodass sie nur aktive Erkrankungen liefert: `"where": [{"path": "clinicalStatus.coding.where(code = 'active').exists()"}]`.
2. Schreibe eine Observation-View für das Körpergewicht (LOINC `29463-7`) mit der Spalte `value.ofType(Quantity).value` und verknüpfen Sie sie mit der Patiententabelle.
3. Speichere das Join-Ergebnis mit `df.write.format("delta").save(...)` als eigene Delta-Tabelle und fragen Sie diese mit DuckDB ab.
4. Erzeuge 1.000 Personen und beobachte, wie sich der Zeitvergleich aus Abschnitt 7 verändert.

Zum Weiterlesen: [Pathling: Abfragen ausführen](https://pathling.csiro.au/docs/libraries/running-queries) · [SQL on FHIR](https://build.fhir.org/ig/HL7/sql-on-fhir/) · [DuckDB-Erweiterung `delta`](https://duckdb.org/docs/stable/core_extensions/delta)